# Heston stochastic volatility: C++ pricing engine, validation and calibration

This notebook drives the C++ engine of `scripts/quant_engine` from Python to

1. build the risk-free discount curve from the **official ECB euro area AAA yield curve** (Svensson parameters);
2. validate the Black-Scholes finite-difference solver (Crank-Nicolson with Rannacher start, PSOR for American exercise) against closed-form prices and a binomial tree;
3. validate the Heston Fourier pricer against the published benchmark of Fang and Oosterlee (2008) and an independent SciPy quadrature;
4. study the bias and statistical error of the Quadratic-Exponential Monte Carlo scheme (Andersen, 2008) and compare C++ with vectorised NumPy;
5. analyse the implied volatility surface generated by Heston and the role of each parameter;
6. calibrate the model to a noisy volatility surface and measure parameter identifiability;
7. compute the early-exercise boundary and premium of American puts using the ECB rate.

**Requirements.** Python 3.10+ and the compiled extension: from the repository root run
`python -m pip install -r scripts/quant_engine/requirements.txt` and `python -m pip install -e scripts/quant_engine`
(see the project README for compiler set-up on Windows, macOS and Linux). In VS Code, select the virtual environment as the notebook kernel.

**Data.** With `DATA_MODE = "official"` (default) the ECB Svensson parameters are downloaded from the ECB Data Portal and cached in `data/raw/ecb/`.
Set the environment variable `QUANT_ENGINE_DATA_MODE=synthetic` to run offline with illustrative parameters (not official data).

**Conventions.** Rates and dividend yields are continuously compounded; maturities are year fractions; option prices are per unit of underlying with $S_0 = 100$.
The ECB curve is a sovereign AAA curve; it is used here as a proxy for the euro risk-free rate.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository root from the notebook folder (used for data and output paths).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "quant_engine").is_dir())
try:
    import quant_engine
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "quant_engine"))
    import quant_engine
from quant_engine import black_scholes as bs, curves, data, heston, synthetic

core = quant_engine.require_cpp()  # raises a clear message if the C++ extension is not built

DATA_MODE = os.environ.get("QUANT_ENGINE_DATA_MODE", "official")  # "official" (ECB) or "synthetic" (offline)
SEED = 20240531                                                   # fixed seed for every simulation below
OUTPUT_DIR = ROOT / "outputs" / "heston_pricing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"quant_engine {quant_engine.__version__} | C++ module: {Path(core.__file__).name} | data mode: {DATA_MODE}")

## 1. Risk-free curve from the ECB AAA yield curve

The ECB estimates its daily yield curve with the Svensson (1994) model:

$$ y(m) = \beta_0 + \beta_1 \frac{1-e^{-m/\tau_1}}{m/\tau_1} + \beta_2\left(\frac{1-e^{-m/\tau_1}}{m/\tau_1} - e^{-m/\tau_1}\right) + \beta_3\left(\frac{1-e^{-m/\tau_2}}{m/\tau_2} - e^{-m/\tau_2}\right), $$

with $\beta_i$ in percent and $\tau_i$ in years. We take the last available business day of December 2025 as valuation date.

In [ ]:
if DATA_MODE == "official":
    svensson = data.load_ecb_svensson_parameters("2025-12-01", "2025-12-31")
else:
    svensson = synthetic.svensson_parameters()
valuation_date = svensson.index[-1]
curve = curves.SvenssonCurve.from_series(svensson.iloc[-1])
print("Source:", svensson.attrs.get("source"))
print("Valuation date:", f"{valuation_date:%Y-%m-%d}")
display(svensson.iloc[[-1]].T.rename(columns=lambda d: f"{d:%Y-%m-%d}"))

m = np.linspace(0.25, 30, 300)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(m, 100 * curve.zero_rate(m), label="zero-coupon rate")
ax.plot(m, 100 * curve.forward_rate(m), "--", label="instantaneous forward rate")
ax.set(xlabel="maturity (years)", ylabel="% (continuous compounding)",
       title=f"Euro area AAA curve, {valuation_date:%d %b %Y}")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ecb_curve.png", dpi=150)

key = [1 / 12, 0.25, 0.5, 1, 2, 5, 10, 30]
display(pd.DataFrame({"zero rate (%)": 100 * curve.zero_rate(key), "discount factor": curve.discount(key)},
                     index=pd.Index(key, name="maturity (years)")))

## 2. Black-Scholes engines: closed form, Crank-Nicolson and binomial tree

European options are priced in closed form and with the finite-difference solver; the American put, which has no closed form,
is compared with a 20,000-step Cox-Ross-Rubinstein tree (NumPy reference). The rate is the ECB 1-year zero rate.

In [ ]:
S0, K, T, sigma, q = 100.0, 100.0, 1.0, 0.20, 0.0
r1 = float(curve.zero_rate(T))
rows = {}
for kind in ("call", "put"):
    exact = core.bs_price_greeks(S0, K, T, r1, q, sigma, kind)["price"]
    fd = core.bs_finite_difference(S0, K, T, r1, q, sigma, kind, False, 1600, 1600)["price"]
    rows[f"European {kind}"] = {"reference": exact, "Crank-Nicolson": fd, "abs. difference": abs(fd - exact)}

t0 = time.perf_counter()
american = core.bs_finite_difference(S0, K, T, r1, q, sigma, "put", True, 1600, 1600)
t_fd = time.perf_counter() - t0
t0 = time.perf_counter()
tree = bs.crr_price(S0, K, T, r1, q, sigma, n_steps=20000, option_type="put", american=True)
t_tree = time.perf_counter() - t0
rows["American put"] = {"reference": tree, "Crank-Nicolson": american["price"],
                        "abs. difference": abs(american["price"] - tree)}
print(f"r(1y) = {100 * r1:.4f}% | CN 1600x1600 with PSOR: {t_fd:.2f} s | CRR tree (NumPy): {t_tree:.2f} s")
print("Reference: closed form for European options, CRR tree for the American put.")
display(pd.DataFrame(rows).T)

In [ ]:
# Empirical order of convergence of the Crank-Nicolson solver (European put, M = N).
exact = core.bs_price_greeks(S0, K, T, r1, q, sigma, "put")["price"]
grid = np.array([100, 200, 400, 800, 1600, 3200])
err = np.array([abs(core.bs_finite_difference(S0, K, T, r1, q, sigma, "put", False, int(n), int(n))["price"] - exact)
                for n in grid])
order = -np.polyfit(np.log(grid), np.log(err), 1)[0]
display(pd.DataFrame({"nodes": grid, "abs. error": err}).set_index("nodes"))
print(f"Estimated order of convergence: {order:.2f} (theory: 2)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(grid, err, "o-", label="Crank-Nicolson error")
ax.loglog(grid, err[0] * (grid[0] / grid) ** 2, "k--", label="slope -2")
ax.set(xlabel="space and time nodes", ylabel="absolute error", title="Convergence to the closed form")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fd_convergence.png", dpi=150)

## 3. Heston Fourier pricer: published benchmark

Fang and Oosterlee (2008, Table 4) price an at-the-money call with $S_0=K=100$, $T=1$, $r=q=0$,
$v_0=0.0175$, $\kappa=1.5768$, $\theta=0.0398$, $\sigma=0.5751$, $\rho=-0.5711$ and report the reference value 5.785155450.
The Feller condition $2\kappa\theta > \sigma^2$ is violated, so the variance can reach zero: a demanding test case.
The C++ pricer integrates the "little Heston trap" characteristic function (Albrecher et al., 2007) with composite Gauss-Legendre panels;
the reference uses `scipy.integrate.quad` on an independent NumPy implementation.

In [ ]:
fo = heston.FANG_OOSTERLEE_PARAMS
t0 = time.perf_counter()
for _ in range(100):
    cpp_price = heston.price(100.0, [100.0], 1.0, 0.0, 0.0, fo)[0]
t_cpp = (time.perf_counter() - t0) / 100
t0 = time.perf_counter()
quad_price = heston.call_price_reference(100.0, 100.0, 1.0, 0.0, 0.0, fo)
t_quad = time.perf_counter() - t0
published = heston.FANG_OOSTERLEE_REFERENCE_CALL
with pd.option_context("display.float_format", "{:.10f}".format):
    display(pd.DataFrame({
        "price": [published, cpp_price, quad_price],
        "difference vs published": [0.0, cpp_price - published, quad_price - published],
        "time per price (ms)": [np.nan, 1e3 * t_cpp, 1e3 * t_quad],
    }, index=["Fang & Oosterlee (2008)", "C++ Gauss-Legendre", "SciPy quad (NumPy)"]))
print(f"Feller ratio 2*kappa*theta/sigma^2 = {fo.feller_ratio:.3f}; speed-up C++ vs SciPy: {t_quad / t_cpp:.0f}x")

## 4. Monte Carlo with the QE scheme: discretisation bias and statistical error

For each time step the variance is sampled with Andersen's Quadratic-Exponential scheme ($\psi_c = 1.5$) and $\ln S$ with his
central discretisation. Paths run in blocks with independent xoshiro256** streams, so results are identical for any number of threads.
The error bars are 95% confidence intervals ($\pm 1.96$ standard errors).

In [ ]:
strikes = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
exact = heston.price(100.0, strikes, 1.0, 0.0, 0.0, fo)
rows = []
for n_steps in (2, 4, 8, 16, 32, 64, 128):
    t0 = time.perf_counter()
    mc = heston.mc_price(100.0, strikes, 1.0, 0.0, 0.0, fo, n_steps=n_steps, n_paths=1_000_000, seed=SEED)
    elapsed = time.perf_counter() - t0
    for k, strike in enumerate(strikes):
        rows.append({"steps per year": n_steps, "strike": strike, "MC price": mc["call"][k],
                     "std. error": mc["call_se"][k], "bias": mc["call"][k] - exact[k], "seconds": elapsed})
mc_table = pd.DataFrame(rows)
display(mc_table[mc_table["strike"] == 100.0].set_index("steps per year"))

fig, ax = plt.subplots(figsize=(7, 4))
for strike in (80.0, 100.0, 120.0):
    sub = mc_table[mc_table["strike"] == strike]
    ax.errorbar(1 / sub["steps per year"], sub["bias"], yerr=1.96 * sub["std. error"], marker="o", capsize=3, label=f"K = {strike:.0f}")
ax.axhline(0.0, color="k", lw=0.8)
ax.set_xscale("log")
ax.set(xlabel="time step (years)", ylabel="MC price - Fourier price", title="QE scheme: bias vs time step (1,000,000 paths)")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "qe_bias.png", dpi=150)

In [ ]:
# Speed: the same QE algorithm in C++ (1 thread / all threads) and in vectorised NumPy.
n_paths, n_steps = 200_000, 50
timings = {}
t0 = time.perf_counter(); core.heston_mc(100.0, strikes, 1.0, 0.0, 0.0, *fo.as_tuple(), n_steps, n_paths, SEED, 1)
timings["C++ (1 thread)"] = time.perf_counter() - t0
t0 = time.perf_counter(); core.heston_mc(100.0, strikes, 1.0, 0.0, 0.0, *fo.as_tuple(), n_steps, n_paths, SEED, 0)
timings[f"C++ ({os.cpu_count()} threads)"] = time.perf_counter() - t0
t0 = time.perf_counter(); ref_mc = heston.mc_price_numpy(100.0, strikes, 1.0, 0.0, 0.0, fo, n_steps, n_paths, SEED)
timings["NumPy (vectorised)"] = time.perf_counter() - t0
speed = pd.Series(timings, name="seconds").to_frame()
speed["speed-up vs NumPy"] = timings["NumPy (vectorised)"] / speed["seconds"]
display(speed)

## 5. Implied volatility surface generated by Heston

Illustrative, equity-index-like parameters. Each maturity is discounted with the ECB zero rate of the same maturity;
the smile is plotted against log-moneyness $\ln(K/F)$ with $F = S_0 e^{(r-q)T}$.

In [ ]:
base = heston.HestonParams(v0=0.04, kappa=1.5, theta=0.05, sigma=0.7, rho=-0.7)
S0, q = 100.0, 0.0
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for T in (1 / 12, 0.25, 1.0, 2.0, 5.0):
    rT = float(curve.zero_rate(T))
    F = S0 * np.exp((rT - q) * T)
    k = np.linspace(-0.5, 0.35, 60) * np.sqrt(max(T, 0.25))
    K = F * np.exp(k)
    iv = heston.implied_vols(heston.price(S0, K, T, rT, q, base), S0, K, T, rT, q)
    axes[0].plot(k, 100 * iv, label=f"T = {T:.2f}")
axes[0].set(title="Smile by maturity", xlabel="ln(K/F)", ylabel="implied volatility (%)")
axes[0].legend(fontsize=8)

T, rT = 1.0, float(curve.zero_rate(1.0))
K = S0 * np.exp(rT * T) * np.exp(np.linspace(-0.5, 0.35, 60))
for rho in (-0.9, -0.5, 0.0, 0.5):
    p = heston.HestonParams(base.v0, base.kappa, base.theta, base.sigma, rho)
    axes[1].plot(np.log(K / (S0 * np.exp(rT))), 100 * heston.implied_vols(heston.price(S0, K, T, rT, q, p), S0, K, T, rT, q), label=f"rho = {rho}")
axes[1].set(title="Correlation drives the skew (T = 1)", xlabel="ln(K/F)")
axes[1].legend(fontsize=8)
for s in (0.2, 0.5, 0.9):
    p = heston.HestonParams(base.v0, base.kappa, base.theta, s, base.rho)
    axes[2].plot(np.log(K / (S0 * np.exp(rT))), 100 * heston.implied_vols(heston.price(S0, K, T, rT, q, p), S0, K, T, rT, q), label=f"sigma = {s}")
axes[2].set(title="Vol of vol drives the curvature (T = 1)", xlabel="ln(K/F)")
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "heston_smiles.png", dpi=150)

## 6. Calibration to a noisy volatility surface

No official source publishes free option quotes, so we test the calibrator in a controlled experiment: a "market" surface is generated
from known parameters and perturbed with Gaussian noise of 0.2 volatility points (a typical half bid-ask spread for liquid index options).
The objective minimises vega-weighted price errors with `scipy.optimize.least_squares`; every evaluation prices the whole surface in C++.
With licensed market quotes, replace `market_vols` and the strikes/maturities with the observed ones.

In [ ]:
true = heston.HestonParams(v0=0.035, kappa=2.0, theta=0.045, sigma=0.6, rho=-0.65)
mats = np.array([1 / 12, 0.25, 0.5, 1.0, 2.0])
x = np.linspace(-1.5, 1.5, 11)
T_grid = np.repeat(mats, x.size)
r_grid = curve.zero_rate(T_grid)
K_grid = S0 * np.exp(r_grid * T_grid) * np.exp(np.tile(x, mats.size) * 0.2 * np.sqrt(T_grid))
clean = heston.implied_vols(heston.price(S0, K_grid, T_grid, r_grid, q, true), S0, K_grid, T_grid, r_grid, q)
rng = np.random.default_rng(SEED)
market_vols = clean + rng.normal(0.0, 0.002, clean.size)

start = heston.HestonParams(v0=0.06, kappa=1.0, theta=0.06, sigma=0.3, rho=-0.2)
t0 = time.perf_counter()
fit = heston.calibrate(S0, K_grid, T_grid, r_grid, q, market_vols, start)
t_cal = time.perf_counter() - t0
names = ["v0", "kappa", "theta", "sigma", "rho"]
display(pd.DataFrame({"true": true.as_tuple(), "start": start.as_tuple(), "calibrated": fit.params.as_tuple()}, index=names))
print(f"least_squares: {fit.n_evaluations} objective evaluations ({K_grid.size} options each, plus finite-difference "
      f"Jacobians) in {t_cal:.2f} s; success = {fit.success}")
print(f"Implied-vol RMSE: {100 * fit.rmse_vol:.3f} vol points (noise s.d. 0.200); max abs error {100 * fit.max_abs_vol_error:.3f}")

In [ ]:
fitted_vols = heston.implied_vols(heston.price(S0, K_grid, T_grid, r_grid, q, fit.params), S0, K_grid, T_grid, r_grid, q)
fig, axes = plt.subplots(1, mats.size, figsize=(16, 3.4), sharey=True)
for ax, T in zip(axes, mats):
    sel = T_grid == T
    ax.plot(x, 100 * market_vols[sel], "o", ms=4, label="market (noisy)")
    ax.plot(x, 100 * fitted_vols[sel], "-", label="calibrated")
    ax.plot(x, 100 * clean[sel], ":", color="k", label="true")
    ax.set(title=f"T = {T:.2f}", xlabel="standardised moneyness")
axes[0].set_ylabel("implied volatility (%)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration_fit.png", dpi=150)

# Identifiability: repeat the experiment with independent noise draws.
draws = []
for rep in range(10):
    noisy = clean + np.random.default_rng(SEED + 1 + rep).normal(0.0, 0.002, clean.size)
    draws.append(heston.calibrate(S0, K_grid, T_grid, r_grid, q, noisy, start).params.as_tuple())
draws = pd.DataFrame(draws, columns=names)
dispersion = draws.std(ddof=1) / np.abs(np.array(true.as_tuple()))
display(pd.DataFrame({"true": true.as_tuple(), "mean": draws.mean(), "std. dev.": draws.std(ddof=1),
                      "relative dispersion": dispersion}, index=names))
print(f"Least identified parameter: {dispersion.idxmax()} (largest relative dispersion across 10 noise draws)")

## 7. American puts: early-exercise boundary and premium

The PSOR solver returns the critical stock price below which immediate exercise is optimal. The premium over the European put
depends on the interest rate: with $r \le 0$ (and no dividends) early exercise is never optimal. The dashed line marks the ECB 1-year rate.

In [ ]:
sigma, K, T = 0.25, 100.0, 1.0
r1 = float(curve.zero_rate(T))
rate_for_boundary = max(r1, 0.01)  # the boundary is only defined when r > 0
fd = core.bs_finite_difference(100.0, K, T, rate_for_boundary, 0.0, sigma, "put", True, 800, 800)
rates = np.linspace(-0.01, 0.06, 15)
premium = [core.bs_finite_difference(100.0, K, T, r, 0.0, sigma, "put", True, 800, 800)["price"]
           - core.bs_price_greeks(100.0, K, T, r, 0.0, sigma, "put")["price"] for r in rates]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fd["tau_grid"], fd["exercise_boundary"])
axes[0].set(xlabel="time to maturity (years)", ylabel="critical stock price",
            title=f"Early-exercise boundary (r = {100 * rate_for_boundary:.2f}%, sigma = {sigma:.0%})")
axes[1].plot(100 * rates, premium, "o-")
axes[1].axvline(100 * r1, ls="--", color="gray", label="ECB 1y zero rate")
axes[1].set(xlabel="interest rate (%)", ylabel="American - European put", title="Early-exercise premium (S0 = K = 100, T = 1)")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "american_put.png", dpi=150)

## 8. Summary

- The Crank-Nicolson solver converges with order 2 to the closed form and matches the binomial tree for American puts.
- The C++ Heston pricer reproduces the Fang-Oosterlee benchmark to about $10^{-8}$ and an independent quadrature to $10^{-10}$ or better, orders of magnitude faster.
- The QE scheme shows a small discretisation bias that vanishes as the time step shrinks, while statistical errors scale as $1/\sqrt{N}$.
- Calibration recovers the smile within the noise level. The mean-reversion speed $\kappa$ is typically the least identified parameter,
  because $\kappa$ and $\theta$ can partly offset each other in the term structure of implied variance.

Figures are saved in `outputs/heston_pricing/`.

### References

- Albrecher, H., Mayer, P., Schoutens, W. and Tistaert, J. (2007). The little Heston trap. *Wilmott Magazine*, January, 83-92.
- Andersen, L. (2008). Simple and efficient simulation of the Heston stochastic volatility model. *Journal of Computational Finance*, 11(3), 1-42.
- Blackman, D. and Vigna, S. (2021). Scrambled linear pseudorandom number generators. *ACM Transactions on Mathematical Software*, 47(4).
- European Central Bank. Euro area yield curves, methodology and data: https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_area_yield_curves/html/index.en.html
- Fang, F. and Oosterlee, C. W. (2008). A novel pricing method for European options based on Fourier-cosine series expansions. *SIAM Journal on Scientific Computing*, 31(2), 826-848.
- Heston, S. L. (1993). A closed-form solution for options with stochastic volatility. *Review of Financial Studies*, 6(2), 327-343.
- Rannacher, R. (1984). Finite element solution of diffusion problems with irregular data. *Numerische Mathematik*, 43, 309-327.
- Svensson, L. E. O. (1994). Estimating and interpreting forward interest rates: Sweden 1992-1994. NBER Working Paper 4871.